# Lab 2 — Getting the time axis right
### EP0418 · Topic 3 · 2 practical hours

**You need:** `ep0418.py` and `sg_taxi_hourly.csv` beside this notebook.

---
This lab uses real Singapore taxi availability pulled from `data.gov.sg`. It is a
live government feed, and it does something that will break your analysis if you
do not notice it.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). In its chat box, click the **sliders icon** next to **+**, choose **Configure**, paste the text below and save. Leave the mode on **Default**. Colab keeps the instructions with this notebook, so every answer follows the course rules.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

CSV = "sg_taxi_hourly.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows")
raw.head()

## Task 1 — Look at the timestamps before anything else

Run the cell below and read the timestamps carefully.

In [ ]:
print(raw.timestamp.head(6).to_string(index=False))
print()
print("gaps between consecutive readings:")
print(raw.timestamp.diff().value_counts().head(5).to_string())

### They are not on the hour

`23:59:48`, then `00:59:50`, then `01:59:53`. The API returns the moment the
reading was actually taken, not a tidy clock tick.

This is **irregular sampling**, and here is why it matters immediately:

If you want "the same hour last week" and you write `.shift(168)`, you are
shifting by **168 rows** — and rows are not evenly spaced in time. You will land
somewhere near last week, off by an unpredictable amount, and your comparison
will be quietly meaningless.

Fix the grid first. Then shift.

## Task 2 — Put it on a regular hourly grid

Prompt your assistant with something like:

> I have a DataFrame `raw` with a `timestamp` column and a `taxi_count` column.
> The timestamps are irregular. Resample onto a regular hourly grid, taking the
> mean within each hour. Do not drop empty hours — leave them as NaN. Then show me
> the row count before and after, and how many hours are empty.

Note the constraint in that last sentence. Without it you will very likely get
code that quietly discards the gaps.

In [ ]:
# TODO(you): make `s` a regular hourly Series of taxi_count
s = ...

# ---- verification contract — do not edit ----
check.regular_cadence(s, "1h")
check.report()

## Task 3 — How many hours are actually missing?

Resampling does not invent data. It creates the empty slots so you can *see* what
was never there.

> I have a pandas Series `s` with an hourly DatetimeIndex. Print how many values
> are NaN and what fraction of the total that is.

In [ ]:
# TODO(you): print how many slots in `s` are NaN, and what fraction that is
...

## Task 4 — Now the shift means what you think it means

With a regular grid, `shift(168)` really is one week (7 days × 24 hours).

**MAE** (mean absolute error) is the average size of the miss: for each hour,
how far the guess was from the truth, ignoring whether it was too high or too
low, averaged over all hours.

> `s` is an hourly pandas Series. Build `snaive`, which is `s` shifted by 168
> hours. Then compute the mean absolute error between `s` and `snaive`, using
> only rows where both exist, and call it `mae_snaive`.

In [ ]:
# TODO(you): build `snaive` = the value from 168 hours earlier
snaive = ...

# TODO(you): compute the mean absolute error between s and snaive,
#            using only rows where BOTH exist
mae_snaive = ...

if filled(snaive, mae_snaive):
    print(f"seasonal-naive MAE: {mae_snaive:,.1f}")
    print(f"mean taxi count   : {s.mean():,.0f}")

# ---- verification contract — do not edit ----
# The baseline's MAE is compared with the series' standard deviation, which is
# roughly the error of always guessing the average. The report calls the first
# number "Model" because the check is general; here it is your baseline.
if filled(s):
    check.compare_to_baseline(mae_snaive, s.std(), metric="MAE")
check.report()

## Task 5 — See the two cycles

Plot the mean taxi count by hour of day, and by day of week.

In [ ]:
# TODO(you): two plots — mean by hour-of-day, and mean by day-of-week
...

---
## Going further (optional): a real market's time axis

For the adventurous. Real hourly share prices have their own time-axis problems.
This uses **yfinance**, a free, unofficial library for Yahoo Finance data, meant for
personal, educational use (pypi.org/project/yfinance).

Set `TRY_LIVE = True` and run the cell. It downloads a month of hourly prices for
ES3.SI (an ETF of Singapore shares) once, saves them, and reuses the saved copy, so
the class does not hit Yahoo's limits. If Yahoo refuses, try again later.

Then answer:
1. How many rows does each trading day have? Where are the gaps?
2. What time zone are the timestamps in?
3. Should you put this onto a regular hourly timetable, as you did with the taxi
   feed? Why not?

In [ ]:
TRY_LIVE = False      # set to True to try it

if TRY_LIVE:
    import os
    import yfinance as yf          # if this fails: run  !pip install yfinance  first
    if not os.path.exists("es3_hourly.csv"):
        yf.Ticker("ES3.SI").history(period="1mo", interval="1h").to_csv("es3_hourly.csv")
    live = pd.read_csv("es3_hourly.csv", index_col=0)
    live.index = pd.to_datetime(live.index, utc=True).tz_convert("Asia/Singapore")
    print("time zone:", live.index.tz)
    print(live.groupby(live.index.date).size().tail(10))   # rows per trading day

**My answers:**

1.
2.
3.

---
## Read before you run

An assistant was asked to put the taxi feed onto a regular hourly grid and “make sure there are no missing values”. It wrote:

**Do not run it yet.** Read it, answer the two questions in the cell below,
and only then remove the `#` and run it to see whether you were right.

What does the second line do to the 39 empty hours, and why does that matter for anything computed from `s` afterwards?

Write down what happens to the minimum and the mean of `s`.

In [ ]:
# --- AI-generated. Read it first. ---
# s = raw.set_index("timestamp").taxi_count.resample("h").mean()
# s = s.fillna(0)

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Before you leave

Add to `verification_log.md`:

- What did your assistant produce for Task 2 on the first attempt? Did it drop the
  empty hours, or keep them?
- How many hours were missing from the feed, and did that surprise you?

**One thing to notice:** the minimum in this data is about 24 taxis available
across all of Singapore. That is not plausible — it is a glitch in the feed. It is
real, it is not something we added, and you will have to decide what to do with it
in your project.